# GA4 BigQuery Ecommerce Case Study

This notebook is a step-by-step exploration of Google's public GA4 ecommerce BigQuery dataset for the Google Merchandise Store. The goal is not only to make charts. The goal is to show how raw GA4 event rows become business metrics that an ecommerce team could act on.

**Dataset:** `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`  
**Coverage:** 2020-11-01 to 2021-01-31  
**Official source:** https://developers.google.com/analytics/bigquery/web-ecommerce-demo-dataset

Google describes this as an obfuscated sample dataset. That makes it useful for learning GA4 and BigQuery SQL, but business conclusions should be framed as patterns in the sample data rather than claims about the real Google Merchandise Store.

## How to Read This Notebook

Each section follows the same pattern:

1. Business question
2. Why the question improves the business
3. GA4 fields or events needed
4. BigQuery SQL pattern learned
5. Interpretation notes

The important SQL concepts are `_TABLE_SUFFIX`, `UNNEST(event_params)`, `UNNEST(items)`, session reconstruction, conditional aggregation, and `SAFE_DIVIDE`.

## Setup

Run this notebook in an environment that can access BigQuery, such as Google Colab, Vertex AI Workbench, or a local Jupyter environment authenticated with Google Cloud. This copy is configured for `big-query-project-505612`. BigQuery public datasets still require a project for billing/accounting, even when the data is public.

In [1]:
import pandas as pd

try:
    from google.cloud import bigquery
except ImportError:
    bigquery = None

PROJECT_ID = "big-query-project-505612"
TABLE_PATTERN = "`bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`"
START_SUFFIX = "20201101"
END_SUFFIX = "20210131"

def run_query(sql: str) -> pd.DataFrame:
    if PROJECT_ID == "your-gcp-project-id" or bigquery is None:
        print("Set PROJECT_ID and install google-cloud-bigquery to execute this query. SQL shown below.\n")
        print(sql)
        return pd.DataFrame()
    client = bigquery.Client(project=PROJECT_ID)
    return client.query(sql).to_dataframe()


## Live BigQuery results

The reusable output layer for this case study is `data/live_ga4_summary.json`. It was generated from BigQuery using `scripts/build_live_ga4_outputs.py`, then reused by the notebook and static dashboard.

- Event rows queried: **4,295,584**
- Users: **270,154**
- Sessions: **360,129**
- Purchases: **5,692**
- Revenue in the sample export: **$362,165**
- Session-to-purchase rate: **1.58%**

These numbers should be read as patterns in the public, obfuscated GA4 sample dataset, not as audited commercial performance for the live Google Merchandise Store.


In [ ]:
import json
from pathlib import Path

summary_candidates = [
    Path('data/live_ga4_summary.json'),
    Path('projects/ecommerce-growth-diagnostics-lab/data/live_ga4_summary.json'),
]
SUMMARY_PATH = next(path for path in summary_candidates if path.exists())
live_summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))

pd.Series(live_summary['kpis'])


## Finding highlights

- **Traffic scale is large enough for funnel diagnosis**: 360,129 sessions and 5,692 purchases across the sample period. The data supports conversion diagnostics, not just top-line traffic reporting.
- **The largest funnel leak identifies the first optimization target**: View item to Add to cart loses 80.3% of sessions. Prioritize the journey step with the largest loss before scaling acquisition spend.
- **Revenue concentration points to channel prioritization**: google / organic contributes $95,775 in revenue. Compare high-revenue channels against high-efficiency channels before changing budget mix.
- **Efficient channels may be under-scaled**: (data deleted) / (data deleted) converts at 3.1%. A smaller but efficient channel can justify lifecycle, SEO, or paid test expansion.
- **Merchandising should separate attention from conversion**: Apparel leads category revenue with $171,727. High-revenue categories deserve visibility, while high-view low-purchase categories need offer or product-page review.


In [ ]:
pd.DataFrame(live_summary['findings'])


## Optimization Strategy Layer

Descriptive metrics tell us what happened. The stronger business case is to ask which observable session, item, channel, and journey features are associated with better or worse outcomes, then translate those patterns into testable actions.

This layer uses the generated `data/live_ga4_summary.json` output so the notebook can show the current live BigQuery result first, while the SQL sections below explain how the metrics are built from GA4 events.


### Feature association analysis

This table compares session-level features against the sitewide purchase baseline. The goal is not to claim causality. The goal is to identify useful segmentation cuts for acquisition quality, merchandising, remarketing, localization, and UX follow-up.


In [ ]:
feature_associations = pd.DataFrame(live_summary['optimization_analysis']['feature_associations'])
feature_associations[
    [
        'feature_group',
        'feature_value',
        'sessions',
        'conversion_rate',
        'lift_vs_baseline',
        'revenue_per_session',
        'business_use',
    ]
].head(12)


### Step-specific funnel driver analysis

A single conversion rate hides where the business problem lives. Step-specific funnel driver analysis compares segments separately for `view_item -> add_to_cart`, `add_to_cart -> begin_checkout`, and `begin_checkout -> purchase`, so each optimization idea can target a specific customer decision.


In [ ]:
step_drivers = live_summary['optimization_analysis']['step_drivers']
step_driver_rows = []
for step_key, rows in step_drivers.items():
    step_driver_rows.extend(rows)

pd.DataFrame(step_driver_rows)[
    [
        'step_name',
        'feature_group',
        'feature_value',
        'eligible_sessions',
        'converted_sessions',
        'step_conversion_rate',
        'lift_vs_baseline',
        'strategic_read',
    ]
].head(15)


### Product opportunity matrix

The product opportunity matrix separates attention from conversion. High-view, low-conversion categories are candidates for product-page or offer fixes; low-view, high-conversion categories are candidates for more visibility.


In [ ]:
product_matrix = pd.DataFrame(live_summary['optimization_analysis']['product_opportunity_matrix'])
product_matrix[
    ['name', 'views', 'purchases', 'revenue', 'viewToPurchase', 'quadrant', 'recommended_action']
]


### Channel quality scorecard

Channel strategy should compare quality and scale. This scorecard combines conversion rate, revenue per session, and cart progression to separate channels worth protecting or testing from channels that need landing-page or audience-quality work first.


In [ ]:
channel_scorecard = pd.DataFrame(live_summary['optimization_analysis']['channel_quality_scorecard'])
channel_scorecard[
    ['name', 'sessions', 'purchases', 'conv', 'revenue_per_session', 'quality_score', 'strategic_read']
]


### Experiment backlog

The final case-study move is to turn analysis into a prioritized experiment backlog. Each idea has a hypothesis, segment, primary metric, and measurement plan, which makes the recommendation testable instead of just descriptive.


In [ ]:
experiment_backlog = pd.DataFrame(live_summary['optimization_analysis']['experiment_backlog'])
experiment_backlog[['priority', 'hypothesis', 'segment', 'primary_metric', 'measurement_plan']]


## Rebuild the live output layer

The notebook can run ad hoc BigQuery queries cell by cell. The portfolio dashboard uses the generated output layer so the public page is fast and reproducible. Rebuild the data layer from the repository root with:

```powershell
.\.venv\Scripts\python.exe projects\ecommerce-growth-diagnostics-lab\scripts\build_live_ga4_outputs.py
```

That command refreshes `data/live_ga4_summary.json`, `data/live_ga4_summary.js`, and CSV extracts for funnel, channel, product category, device, geography, and daily trend summaries.


## GA4 Export Concepts

GA4 exports are event-level tables. Every row is an event, not a session, transaction, or user summary. Some fields are nested records, and some are repeated arrays.

- `event_name` tells us what happened.
- `user_pseudo_id` approximates a browser or device-level user.
- `event_params` stores repeated key/value event attributes such as `ga_session_id`.
- `items` stores repeated product-level data attached to ecommerce events.
- `ecommerce` stores transaction-level values such as purchase revenue.
- `_TABLE_SUFFIX` keeps wildcard-table scans limited to the intended dates.

A major learning objective is knowing when an answer should be event-level, session-level, user-level, or item-level.

## Business Question 1: What is the activity and purchase baseline?

**Why it improves the business:** Before optimizing a funnel, the team needs a baseline: how much traffic exists, how many users are active, how many purchase events occur, and how much revenue appears in the sample.

**GA4 fields/events needed:** `event_date`, `event_name`, `user_pseudo_id`, `ecommerce.purchase_revenue`.

**SQL pattern learned:** wildcard table querying with `_TABLE_SUFFIX`, `COUNTIF`, `COUNT(DISTINCT ...)`, and revenue aggregation.

In [2]:
baseline_sql = f"""
SELECT
  COUNT(*) AS event_count,
  COUNT(DISTINCT user_pseudo_id) AS user_count,
  COUNT(DISTINCT event_date) AS day_count,
  COUNTIF(event_name = 'purchase') AS purchase_events,
  SUM(IFNULL(ecommerce.purchase_revenue, 0)) AS purchase_revenue
FROM {TABLE_PATTERN}
WHERE _TABLE_SUFFIX BETWEEN '{START_SUFFIX}' AND '{END_SUFFIX}';
"""

baseline = run_query(baseline_sql)
baseline


,event_count,user_count,day_count,purchase_events,purchase_revenue
0,4295584,270154,92,5692,362165.0


**Interpretation notes:** This establishes scale. A business stakeholder can use this to decide whether the next level of analysis should focus on traffic growth, conversion efficiency, product mix, or retention.

## Business Question 2: What are users doing before they buy?

**Why it improves the business:** Event frequency shows which parts of the customer journey are heavily used and which are rare. It helps separate normal browsing behavior from commercially meaningful behavior.

**GA4 fields/events needed:** `event_name`, `user_pseudo_id`, and `ga_session_id` extracted from `event_params`.

**SQL pattern learned:** extracting repeated event parameters with `UNNEST(event_params)` and rebuilding a session key.

In [3]:
event_mix_sql = f"""
WITH events AS (
  SELECT
    event_name,
    user_pseudo_id,
    CONCAT(
      user_pseudo_id,
      '-',
      CAST((SELECT value.int_value FROM UNNEST(event_params) WHERE key = 'ga_session_id') AS STRING)
    ) AS session_id
  FROM {TABLE_PATTERN}
  WHERE _TABLE_SUFFIX BETWEEN '{START_SUFFIX}' AND '{END_SUFFIX}'
)
SELECT
  event_name,
  COUNT(*) AS events,
  COUNT(DISTINCT user_pseudo_id) AS users,
  COUNT(DISTINCT session_id) AS sessions,
  SAFE_DIVIDE(COUNT(*), COUNT(DISTINCT user_pseudo_id)) AS events_per_user
FROM events
GROUP BY event_name
ORDER BY events DESC;
"""

event_mix = run_query(event_mix_sql)
event_mix.head(20)


,event_name,events,users,sessions,events_per_user
0,page_view,1350428,269792,333683,5.005441
1,user_engagement,1058721,213004,275517,4.970428
2,scroll,493072,138098,174140,3.570450
3,view_item,386068,61252,77020,6.302945
4,session_start,354970,267116,354857,1.328898
5,first_visit,257462,257314,257400,1.000575
6,view_promotion,190104,102443,128447,1.855705
7,add_to_cart,58543,12545,15188,4.666640
8,begin_checkout,38757,9715,11106,3.989398
9,select_item,31007,13180,15828,2.352580


**Interpretation notes:** The event table helps identify the natural funnel vocabulary. For ecommerce, the key events are usually `view_item`, `add_to_cart`, `begin_checkout`, and `purchase`.

## Business Question 3: Where does the ecommerce funnel lose users?

**Why it improves the business:** Funnel drop-off points point to different operating actions. Weak view-to-cart movement suggests product detail or merchandising issues. Weak cart-to-checkout movement suggests cart friction, price shock, shipping issues, or weak purchase intent. Weak checkout-to-purchase movement suggests payment or checkout UX friction.

**GA4 fields/events needed:** ecommerce event names, `user_pseudo_id`, and session ID.

**SQL pattern learned:** session-level flags, conditional aggregation, window functions, and `SAFE_DIVIDE` for conversion rates.

In [4]:
funnel_sql = f"""
WITH event_sessions AS (
  SELECT
    user_pseudo_id,
    CONCAT(
      user_pseudo_id,
      '-',
      CAST((SELECT value.int_value FROM UNNEST(event_params) WHERE key = 'ga_session_id') AS STRING)
    ) AS session_id,
    event_name
  FROM {TABLE_PATTERN}
  WHERE _TABLE_SUFFIX BETWEEN '{START_SUFFIX}' AND '{END_SUFFIX}'
),
session_flags AS (
  SELECT
    session_id,
    ANY_VALUE(user_pseudo_id) AS user_pseudo_id,
    MAX(IF(event_name = 'session_start', 1, 0)) AS did_session_start,
    MAX(IF(event_name = 'view_item', 1, 0)) AS did_view_item,
    MAX(IF(event_name = 'add_to_cart', 1, 0)) AS did_add_to_cart,
    MAX(IF(event_name = 'begin_checkout', 1, 0)) AS did_begin_checkout,
    MAX(IF(event_name = 'purchase', 1, 0)) AS did_purchase
  FROM event_sessions
  WHERE session_id IS NOT NULL
  GROUP BY session_id
),
funnel AS (
  SELECT 1 AS step_order, 'Session start' AS step, COUNTIF(did_session_start = 1) AS sessions, COUNT(DISTINCT IF(did_session_start = 1, user_pseudo_id, NULL)) AS users FROM session_flags UNION ALL
  SELECT 2, 'View item', COUNTIF(did_view_item = 1), COUNT(DISTINCT IF(did_view_item = 1, user_pseudo_id, NULL)) FROM session_flags UNION ALL
  SELECT 3, 'Add to cart', COUNTIF(did_add_to_cart = 1), COUNT(DISTINCT IF(did_add_to_cart = 1, user_pseudo_id, NULL)) FROM session_flags UNION ALL
  SELECT 4, 'Begin checkout', COUNTIF(did_begin_checkout = 1), COUNT(DISTINCT IF(did_begin_checkout = 1, user_pseudo_id, NULL)) FROM session_flags UNION ALL
  SELECT 5, 'Purchase', COUNTIF(did_purchase = 1), COUNT(DISTINCT IF(did_purchase = 1, user_pseudo_id, NULL)) FROM session_flags
)
SELECT
  step_order,
  step,
  sessions,
  users,
  SAFE_DIVIDE(sessions, LAG(sessions) OVER (ORDER BY step_order)) AS step_conversion_rate,
  SAFE_DIVIDE(sessions, FIRST_VALUE(sessions) OVER (ORDER BY step_order)) AS total_conversion_rate
FROM funnel
ORDER BY step_order;
"""

funnel = run_query(funnel_sql)
funnel


,step_order,step,sessions,users,step_conversion_rate,total_conversion_rate
0,1,Session start,354857,267116,NaN,1.000000
1,2,View item,77020,61252,0.217045,0.217045
2,3,Add to cart,15188,12545,0.197196,0.042800
3,4,Begin checkout,11106,9715,0.731235,0.031297
4,5,Purchase,4848,4419,0.436521,0.013662


**Interpretation notes:** The largest percentage drop is the first candidate for product, UX, pricing, or lifecycle testing. The business action depends on which step is leaking, not just the total conversion rate.

## Business Question 4: Which products attract interest versus revenue?

**Why it improves the business:** Products with many views but low purchase rates may need better pricing, descriptions, imagery, reviews, or bundling. Products with high purchase rates and low visibility may deserve more promotion.

**GA4 fields/events needed:** `event_name` plus product fields in the repeated `items` record.

**SQL pattern learned:** item-level analysis with `UNNEST(items)`.

In [5]:
product_sql = f"""
WITH item_events AS (
  SELECT
    event_name,
    item.item_id,
    item.item_name,
    item.item_category,
    item.price,
    item.quantity,
    item.item_revenue
  FROM {TABLE_PATTERN},
  UNNEST(items) AS item
  WHERE _TABLE_SUFFIX BETWEEN '{START_SUFFIX}' AND '{END_SUFFIX}'
)
SELECT
  item_category,
  item_name,
  COUNTIF(event_name = 'view_item') AS item_views,
  COUNTIF(event_name = 'add_to_cart') AS cart_adds,
  COUNTIF(event_name = 'purchase') AS purchases,
  SUM(IFNULL(item_revenue, 0)) AS item_revenue,
  SAFE_DIVIDE(COUNTIF(event_name = 'add_to_cart'), COUNTIF(event_name = 'view_item')) AS view_to_cart_rate,
  SAFE_DIVIDE(COUNTIF(event_name = 'purchase'), COUNTIF(event_name = 'view_item')) AS view_to_purchase_rate
FROM item_events
GROUP BY item_category, item_name
ORDER BY item_revenue DESC;
"""

products = run_query(product_sql)
products.head(25)


,item_category,item_name,item_views,cart_adds,purchases,item_revenue,view_to_cart_rate,view_to_purchase_rate
0,Apparel,Google Zip Hoodie F/C,15,3,255,13692.0,0.200000,17.000000
1,Apparel,Google Men's Tech Fleece Grey,252,38,123,9965.0,0.150794,0.488095
2,Apparel,Google Badge Heavyweight Pullover Black,285,32,183,9573.0,0.112281,0.642105
3,Shop by Brand,Super G Unisex Joggers,5,0,269,8947.0,0.000000,53.800000
4,Apparel,Google Crewneck Sweatshirt Green,293,96,174,8118.0,0.327645,0.593857
5,Uncategorized Items,Google Crewneck Sweatshirt Navy,0,0,131,6204.0,NaN,NaN
6,Apparel,Google Men's Puff Jacket Black,188,18,63,6187.0,0.095745,0.335106
7,Apparel,Google Sherpa Zip Hoodie Charcoal,170,49,102,5956.0,0.288235,0.600000
8,Apparel,Google Men's Tech Fleece Vest Charcoal,107,10,77,5549.0,0.093458,0.719626
9,Apparel,Google Women's Puff Jacket Black,76,10,56,5313.0,0.131579,0.736842


**Interpretation notes:** This moves the case study beyond generic site conversion. It gives merchandising and lifecycle teams a ranked list of where attention and revenue diverge.

## Business Question 5: Which traffic, device, and geography segments create value?

**Why it improves the business:** Acquisition and UX decisions should not be based only on traffic volume. A lower-volume segment can be more valuable if it converts better or buys more. Device and geography cuts help identify UX friction and market concentration.

**GA4 fields/events needed:** `traffic_source`, `device.category`, `geo.country`, ecommerce revenue, and session flags.

**SQL pattern learned:** building a reusable session mart, grouping by dimensions, and comparing conversion and revenue side by side.

In [6]:
segment_sql = f"""
WITH event_sessions AS (
  SELECT
    user_pseudo_id,
    CONCAT(
      user_pseudo_id,
      '-',
      CAST((SELECT value.int_value FROM UNNEST(event_params) WHERE key = 'ga_session_id') AS STRING)
    ) AS session_id,
    event_name,
    COALESCE(traffic_source.source, '(direct)') AS source,
    COALESCE(traffic_source.medium, '(none)') AS medium,
    device.category AS device_category,
    geo.country AS country,
    ecommerce.purchase_revenue AS purchase_revenue
  FROM {TABLE_PATTERN}
  WHERE _TABLE_SUFFIX BETWEEN '{START_SUFFIX}' AND '{END_SUFFIX}'
),
session_mart AS (
  SELECT
    session_id,
    ANY_VALUE(user_pseudo_id) AS user_pseudo_id,
    ANY_VALUE(source) AS source,
    ANY_VALUE(medium) AS medium,
    ANY_VALUE(device_category) AS device_category,
    ANY_VALUE(country) AS country,
    MAX(IF(event_name = 'view_item', 1, 0)) AS did_view_item,
    MAX(IF(event_name = 'add_to_cart', 1, 0)) AS did_add_to_cart,
    MAX(IF(event_name = 'purchase', 1, 0)) AS did_purchase,
    SUM(IFNULL(purchase_revenue, 0)) AS revenue
  FROM event_sessions
  WHERE session_id IS NOT NULL
  GROUP BY session_id
)
SELECT
  source,
  medium,
  device_category,
  country,
  COUNT(*) AS sessions,
  SUM(did_view_item) AS product_view_sessions,
  SUM(did_add_to_cart) AS cart_sessions,
  SUM(did_purchase) AS purchase_sessions,
  SUM(revenue) AS revenue,
  SAFE_DIVIDE(SUM(did_purchase), COUNT(*)) AS session_to_purchase_rate,
  SAFE_DIVIDE(SUM(revenue), NULLIF(SUM(did_purchase), 0)) AS average_order_value
FROM session_mart
GROUP BY source, medium, device_category, country
HAVING sessions >= 50
ORDER BY revenue DESC;
"""

segments = run_query(segment_sql)
segments.head(25)


,source,medium,device_category,country,sessions,product_view_sessions,cart_sessions,purchase_sessions,revenue,session_to_purchase_rate,average_order_value
0,google,organic,desktop,United States,28604,6065,1112,302,24336.0,0.010558,80.582781
1,google,organic,mobile,United States,19646,4202,802,242,19437.0,0.012318,80.318182
2,(direct),(none),desktop,United States,21416,4524,895,270,18191.0,0.012607,67.374074
3,(direct),(none),mobile,United States,14540,3029,619,191,14718.0,0.013136,77.057592
4,<Other>,referral,desktop,United States,8896,1921,384,133,13088.0,0.014951,98.406015
5,(data deleted),(data deleted),desktop,United States,5677,1479,348,183,13064.0,0.032235,71.387978
6,shop.googlemerchandisestore.com,referral,desktop,United States,7322,1619,357,130,10332.0,0.017755,79.476923
7,<Other>,<Other>,desktop,United States,13154,2674,495,128,9253.0,0.009731,72.289062
8,shop.googlemerchandisestore.com,referral,mobile,United States,5014,1180,265,99,8556.0,0.019745,86.424242
9,(data deleted),(data deleted),mobile,United States,3988,1001,241,114,7133.0,0.028586,62.570175


**Interpretation notes:** A useful case study compares volume and quality. High sessions with weak conversion suggests optimization opportunity. Low sessions with strong conversion suggests a candidate for budget, SEO, email, or lifecycle expansion.

## Outcome: Translate Analysis Into Business Actions

The final write-up should connect evidence to decisions:

| Finding pattern | Likely business interpretation | Possible action |
|---|---|---|
| High product views, low add-to-cart | Product detail page or offer is not convincing | Improve images, copy, pricing, reviews, shipping clarity |
| High add-to-cart, low checkout | Cart friction or price/shipping surprise | Test cart reminders, clarify shipping, reduce cart-step friction |
| High checkout, low purchase | Checkout UX or payment friction | Audit payment options, errors, form length, trust signals |
| Low traffic, high conversion source | Under-scaled quality audience | Increase controlled spend or lifecycle exposure |
| Mobile conversion trails desktop | Mobile experience may be weaker | Review mobile performance, PDP layout, checkout forms |

This is the core skill the case study should showcase: reason from raw event data to operational priorities without overclaiming what the sample dataset can prove.

## Case Study Summary Template

Use this structure in the portfolio write-up:

1. **Problem:** ecommerce teams need to know where users lose momentum between discovery and purchase.
2. **Data:** GA4 public BigQuery ecommerce export from the Google Merchandise Store sample dataset.
3. **Method:** build event, session, funnel, product, and segment views using BigQuery SQL.
4. **Skills demonstrated:** wildcard tables, repeated records, `UNNEST(event_params)`, `UNNEST(items)`, CTEs, `SAFE_DIVIDE`, funnel metrics, and business interpretation.
5. **Outcome:** an analysis workflow that turns raw GA4 events into a prioritized ecommerce diagnostics readout.
6. **Limitations:** obfuscated public sample data; findings are analytical examples, not verified commercial results.